# FinSight - Feature Engineering Module

Transforms raw banking data into meaningful business features that improve credit risk prediction.

In [1]:
import pandas as pd
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append(os.path.abspath('..'))

from src.data.transformer import FeatureEngineer, DataTransformer
from src.analytics.report_generator import ReportGenerator

## 1. Load Cleaned Dataset

In [2]:
DATA_PATH = '../data/processed/application_train_cleaned.csv'
print(f'Loading data from {DATA_PATH}...')
df = pd.read_csv(DATA_PATH)
print(f'Loaded dataset with shape: {df.shape}')

Loading data from ../data/processed/application_train_cleaned.csv...


Loaded dataset with shape: (307511, 266)


## 2. Initialize Transformers

In [3]:
fe = FeatureEngineer()
dt = DataTransformer(model_dir='../models/')
report_gen = ReportGenerator(output_dir='../reports/')

## 3. Generate Domain-Specific Features

In [4]:
df_feat = fe.generate_financial_ratios(df)
df_feat = fe.generate_family_features(df_feat)
df_feat = fe.generate_employment_features(df_feat)
df_feat = fe.generate_asset_features(df_feat)
df_feat = fe.generate_credit_features(df_feat)
df_feat = fe.generate_risk_features(df_feat)
print(f'Shape after domain feature engineering: {df_feat.shape}')

# Save Feature Dictionary
fe.generate_feature_dictionary('../reports/Feature_Dictionary.md')

Shape after domain feature engineering: (307511, 293)


## 4. Encoding & Scaling

In [5]:
df_encoded = dt.encode_features(df_feat)
print(f'Shape after encoding: {df_encoded.shape}')

df_scaled = dt.scale_features(df_encoded)
print(f'Shape after scaling: {df_scaled.shape}')

Shape after encoding: (307511, 314)


Shape after scaling: (307511, 314)


## 5. Feature Selection

In [6]:
df_final, importance_report = dt.select_features(df_scaled, target_col='TARGET')
print(f'Final Dataset Shape: {df_final.shape}')

# Save Importance Report CSV
importance_report.to_csv('../reports/Engineered_Features.csv', index=False)
print('Saved Engineered_Features.csv')

Final Dataset Shape: (307511, 160)
Saved Engineered_Features.csv


## 6. Generate PDFs & Save Final Dataset

In [7]:
# Generate Feature Engineering Report
report_gen.generate_pdf_report(
    'FinSight Feature Engineering Report',
    ['Feature Engineering Pipeline executed successfully.', 
     f'Final dataset shape: {df_final.shape}',
     f'Number of dropped redundant/low-variance features: {len(importance_report[importance_report.Action == 'Dropped'])}'],
    'Feature_Engineering_Report.pdf'
)

# Generate Correlation Report
top_corrs = importance_report[importance_report.Action == 'Kept'].head(20)
corr_lines = ['Top Correlated Features with TARGET:'] + [f'- {row.Feature}: {row.Reason}' for _, row in top_corrs.iterrows()]
report_gen.generate_pdf_report('FinSight Feature Correlation Report', corr_lines, 'Feature_Correlation_Report.pdf')

# Save Output Dataset
out_path = '../data/processed/application_train_featured.csv'
df_final.to_csv(out_path, index=False)
print(f'Final engineered dataset saved to {out_path}')

Final engineered dataset saved to ../data/processed/application_train_featured.csv
